# MEIKA Vox · transcripción de audios desde Google Drive

Código abierto y gratuito para investigación social y territorial. No necesitas saber programar.

1. En **Entorno de ejecución → Cambiar tipo de entorno de ejecución**, selecciona GPU T4 si está disponible.
2. Pulsa **Entorno de ejecución → Ejecutar todas**. Espera a que termine la instalación.
3. Autoriza el acceso a tu propio Google Drive cuando Google lo solicite.
4. En el panel que aparecerá **al final del cuaderno**, elige una carpeta con audios, idioma y nombre del proyecto.
5. Presiona **Transcribir carpeta**. El panel mostrará el avance por archivos.

**Resultados:** Mi unidad / MEIKA_Vox / Proyectos / nombre_del_proyecto / Transcripciones.

**Privacidad:** los audios no se publican en GitHub; Colab necesita tu autorización para acceder a Drive. Revisa los permisos y las condiciones de Google antes de usar audios sensibles.

**Límites:** Colab gratuito no garantiza GPU ni sesiones permanentes. La separación opcional de hablantes requiere un token de Hugging Face. Toda transcripción automática debe revisarse antes de utilizarse como evidencia.


In [ ]:
import os
import shutil
import subprocess
import sys
import sysconfig
from pathlib import Path

REPO = Path("/content/meika-vox")


def checked(cmd, **kwargs):
    subprocess.run([str(item) for item in cmd], check=True, **kwargs)


checked(["apt-get", "-qq", "update"])
checked(["apt-get", "-qq", "install", "-y", "ffmpeg", "git"])
# Durante las pruebas, instalar la misma rama que contiene la interfaz y
# el código del progreso. Evita mezclar el cuaderno nuevo con el motor de main.
REPO_REF = "feat/colab-simple-user-mode-20261009"
if not (REPO / ".git").exists():
    checked(["git", "clone", "--quiet", "https://github.com/MeikaLab/meika-vox.git", REPO])
checked(["git", "-C", REPO, "fetch", "--quiet", "origin", REPO_REF])
checked(["git", "-C", REPO, "checkout", "--quiet", "--detach", "FETCH_HEAD"])
# Colab ya trae un entorno Python funcional. No crear otro con venv:
# algunas imágenes de Colab no incorporan ensurepip para /usr/bin/python3.
PYTHON = Path(sys.executable)
print(f"Instalando MEIKA Vox para Python {sys.version.split()[0]} ...")
checked([PYTHON, "-m", "pip", "install", "--disable-pip-version-check", "-e", str(REPO) + "[whisperx]"])
VOX = Path(sysconfig.get_path("scripts")) / "meika-vox"
if not VOX.is_file():
    executable = shutil.which("meika-vox")
    if executable:
        VOX = Path(executable)
if not VOX.is_file():
    raise RuntimeError("No se encontró meika-vox tras instalar. Revisa los mensajes anteriores.")
checked([VOX, "doctor", "--strict"])
print("Instalación comprobada. Ahora conecta Google Drive.")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import importlib.util
import os
import re
import subprocess
import unicodedata
from collections import Counter
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, HTML
from html import escape

ROOT = Path("/content/drive/MyDrive").resolve()
OUTPUT_ROOT = ROOT / "MEIKA_Vox" / "Proyectos"
spec = importlib.util.spec_from_file_location("vox_batch", REPO / "scripts/colab_batch.py")
batch = importlib.util.module_from_spec(spec)
spec.loader.exec_module(batch)

display(HTML(
    '<div style="border:1px solid #cadfdc;border-radius:12px;background:#f4f9f8;'
    'padding:18px 22px;color:#1a4146;font-family:Arial,sans-serif">'
    '<div style="font-size:22px;font-weight:700">MEIKA Vox</div>'
    '<div style="font-size:13px;color:#4d7378;margin-top:4px">'
    'Transcripción abierta para investigación social y territorial</div>'
    '<p style="font-size:13px">Elige la carpeta que contiene tus grabaciones. '
    'Los resultados se guardarán en tu propio Google Drive.</p></div>'
))

project_input = widgets.Text(
    value="Mi proyecto", description="Proyecto:",
    placeholder="Nombre de tu estudio",
    layout=widgets.Layout(width="95%")
)
language_input = widgets.Dropdown(
    options=[("Español", "es"), ("Inglés", "en"), ("Portugués", "pt"), ("Francés", "fr")],
    value="es", description="Idioma:"
)
glossary_input = widgets.Text(
    value="", description="Glosario:",
    placeholder="Opcional: archivo .json de tu Drive",
    layout=widgets.Layout(width="95%")
)
diarize_input = widgets.Checkbox(value=False, description="Distinguir hablantes (opcional)")
speaker_info = widgets.HTML(value=(
    "<div style='font-size:13px;line-height:1.6;color:#42646a'>"
    "<b>¿Qué hace?</b> Señala cuándo intervienen distintas voces (Voz 1, Voz 2...). "
    "No sabe sus nombres reales y puede cometer errores en conversaciones grupales.<br>"
    "<b>¿Necesita configuración?</b> Sí: una cuenta gratuita de Hugging Face "
    "y una clave de acceso <code>HF_TOKEN</code>. "
    "No necesitas esta opción para transcribir normalmente."
    "</div>"
))
speaker_setup = widgets.HTML(value=(
    "<div style='border:1px solid #d4e5e2;border-radius:9px;background:#f7fbfa;"
    "padding:12px 16px;font-size:13px;line-height:1.65;color:#264b50'>"
    "<b>Activación inicial: 4 pasos gratuitos</b>"
    "<ol style='margin:7px 0;padding-left:20px'>"
    "<li><a href='https://huggingface.co/join' target='_blank' rel='noopener'>"
    "Crea una cuenta en Hugging Face</a>.</li>"
    "<li><a href='https://huggingface.co/pyannote/speaker-diarization-community-1' "
    "target='_blank' rel='noopener'>Acepta las condiciones del modelo pyannote</a>.</li>"
    "<li><a href='https://huggingface.co/settings/tokens' target='_blank' rel='noopener'>"
    "Genera una clave de lectura (token)</a> y cópiala.</li>"
    "<li>En la barra lateral izquierda de Colab abre <b>Secretos (icono de llave)</b>, "
    "crea <code>HF_TOKEN</code>, pega la clave y activa el acceso del cuaderno.</li>"
    "</ol>"
    "<b>Luego marca esta opción y transcribe.</b> Si no guardaste la clave en Secretos, "
    "MEIKA Vox podrá solicitarla mediante una entrada oculta.<br>"
    "<span style='color:#856322'>No compartas tu token por correo, chat ni GitHub. "
    "Esta función puede tardar más y requiere revisar manualmente los hablantes.</span>"
    "</div>"
))
speaker_setup.layout.display = "none"

def show_speaker_help(change):
    speaker_setup.layout.display = "" if change["new"] else "none"

diarize_input.observe(show_speaker_help, names="value")
force_input = widgets.Checkbox(value=False, description="Volver a transcribir los ya terminados")

# Se incluyen carpetas superiores para permitir procesar subcarpetas por lotes.
print("Buscando carpetas con audio en tu Google Drive…")
audios = batch.discover_audio(ROOT, exclude=OUTPUT_ROOT)
counts = Counter(p.parent for p in audios)
for folder in tuple(counts):
    ancestor = folder.parent
    while ancestor.is_relative_to(ROOT) and ancestor != ROOT:
        counts.setdefault(ancestor, 0)
        ancestor = ancestor.parent
if not counts:
    raise RuntimeError("No hay audios compatibles en tu Drive. Sube una carpeta y reejecuta esta celda.")
choices = [
    (str(p.relative_to(ROOT)), str(p))
    for p in sorted(counts, key=lambda x: str(x).casefold())
]
folder_picker = widgets.Dropdown(options=choices, description="Carpeta:", layout=widgets.Layout(width="95%"))
status = widgets.HTML()
progress = widgets.IntProgress(value=0, min=0, max=1, description="Avance:", layout=widgets.Layout(width="95%"))
progress_message = widgets.Label(value="Esperando inicio del procesamiento.")
motor_message = widgets.HTML(value="<span style='color:#52757a'>Motor pendiente de comprobación.</span>")
start_button = widgets.Button(
    description="Transcribir carpeta", button_style="success", layout=widgets.Layout(width="220px")
)
log = widgets.Output()


def selected_folder():
    folder = Path(folder_picker.value).resolve()
    if not folder.is_relative_to(ROOT):
        raise ValueError("Selecciona una carpeta dentro de tu Drive.")
    return folder


def update_count(change=None):
    folder = selected_folder()
    count = len(batch.discover_audio(folder, exclude=OUTPUT_ROOT))
    warning = (
        "<br><span style='color:#a34d28'>Estás seleccionando toda Mi unidad. "
        "Elige una carpeta específica para evitar procesar grabaciones por accidente.</span>"
        if folder == ROOT else ""
    )
    status.value = f"<b>{count} audios</b> en la carpeta y sus subcarpetas." + warning
    start_button.disabled = folder == ROOT


def project_slug(value):
    simplified = unicodedata.normalize("NFKD", value.strip())
    simplified = "".join(c for c in simplified if not unicodedata.combining(c))
    slug = re.sub(r"[^A-Za-z0-9_-]+", "_", simplified).strip("_")[:64]
    if not slug:
        raise ValueError("Escribe un nombre de proyecto válido.")
    return slug


folder_picker.observe(update_count, names="value")
update_count()


def start_batch(button):
    controls = [start_button, folder_picker, project_input,
                language_input, glossary_input, diarize_input, force_input]
    for control in controls:
        control.disabled = True
    log.clear_output()
    with log:
        try:
            import json

            project_id = project_slug(project_input.value)
            folder = selected_folder()
            if folder == ROOT:
                raise ValueError("Selecciona una carpeta concreta, no toda Mi unidad.")
            output = OUTPUT_ROOT / project_id / "Transcripciones"
            glossary = None
            if glossary_input.value.strip():
                glossary = (ROOT / glossary_input.value.strip()).resolve()
                if not glossary.is_relative_to(ROOT) or not glossary.is_file():
                    raise ValueError("El glosario debe ser un archivo existente dentro de tu Drive.")
                if glossary.suffix.lower() != ".json":
                    raise ValueError("El glosario debe ser un archivo .json.")
            if diarize_input.value:
                from getpass import getpass

                try:
                    from google.colab import userdata
                    token = userdata.get("HF_TOKEN")
                except Exception:
                    token = None
                if not token:
                    print("No se encontró HF_TOKEN en los Secretos de Colab.")
                    print("Si aún no configuraste Hugging Face, revisa los 4 pasos del formulario.")
                    token = getpass("Pega aquí tu token de Hugging Face (entrada oculta): ").strip()
                if not token:
                    raise RuntimeError(
                        "No hay una clave HF_TOKEN. Desmarca 'Distinguir hablantes' "
                        "para transcribir sin ella, o sigue los pasos de configuración."
                    )
                os.environ["HF_TOKEN"] = token
                print("Separación de hablantes habilitada. Comprueba que aceptaste las condiciones de pyannote.")
            runtime = json.loads(
                subprocess.check_output([str(VOX), "doctor", "--strict"], text=True)
            )
            device = "cuda" if runtime["cuda_available"] else "cpu"
            model = "large-v3" if device == "cuda" else "small"
            command = [
                str(VOX),
                "--project-id", project_id,
                "--language", language_input.value,
                "--model", model,
                "--device", device,
                "--batch-size", "4" if device == "cuda" else "1",
            ]
            if glossary is not None:
                command += ["--glossary", str(glossary)]
            if diarize_input.value:
                command.append("--diarize")
            revision = subprocess.check_output(
                ["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True
            ).strip()
            configuration = revision + (batch.fingerprint(glossary) if glossary else "")
            print(f"Proyecto: {project_id} | Motor: {model} | Origen: {folder}")
            print(f"Destino: {output}")
            print("Mantén Colab conectado mientras procesa.")
            motor_message.value = (
                "<b>Motor:</b> GPU / WhisperX large-v3"
                if device == "cuda" else "<b>Motor:</b> CPU / WhisperX small (más lento)"
            )
            progress.value = 0
            progress.bar_style = ""
            progress_message.value = "Preparando los archivos seleccionados..."

            def on_progress(state):
                reviewed = state["completed"] + state["skipped"] + state["failed"]
                progress.max = max(state["total"], 1)
                progress.value = min(reviewed, progress.max)
                if state["event"] == "started":
                    progress_message.value = (
                        f"Procesando archivo {state['index']} de {state['total']}: "
                        + str(state["audio"])
                    )
                elif state["event"] == "finished":
                    progress.bar_style = "warning" if state["failed"] else "success"
                    progress_message.value = (
                        f"Terminados {state['completed']}; ya existentes {state['skipped']}; "
                        f"pendientes por error {state['failed']}."
                    )
                else:
                    progress_message.value = f"Revisados {reviewed} de {state['total']} archivos."
            report = batch.process_folder(
                folder, output, command,
                force=force_input.value,
                configuration=configuration,
                on_progress=on_progress
            )
            if report["failed"]:
                print("Hay audios pendientes: revisa los errores y vuelve a ejecutar.")
            else:
                print(f"Terminado. Revisa en tu Drive: {output.relative_to(ROOT)}")
        except Exception as exc:
            progress.bar_style = "danger"
            progress_message.value = "El proceso se detuvo. Revisa los detalles de ejecución."
            print("No se completó:", exc)
        finally:
            for control in controls:
                control.disabled = False


start_button.on_click(start_batch)
advanced = widgets.Accordion(children=[
    widgets.VBox([
        widgets.HTML("<p>Funciones opcionales. La transcripción normal no exige credenciales adicionales.</p>"),
        widgets.HTML("<b>Separación de hablantes</b>"),
        speaker_info, diarize_input, speaker_setup,
        widgets.HTML("<b>Otras opciones técnicas</b>"),
        glossary_input, force_input
    ])
])
advanced.set_title(0, "Opciones avanzadas (opcional)")
advanced.selected_index = None
start_button.button_style = "info"
start_button.layout = widgets.Layout(width="240px", height="42px")
display(widgets.VBox([
    widgets.HTML("<h3 style='color:#1a4146'>1 · Seleccionar grabaciones</h3>"),
    project_input, folder_picker, status, language_input, advanced,
    widgets.HTML("<h3 style='color:#1a4146'>2 · Transcribir y guardar</h3>"),
    motor_message, start_button, progress, progress_message,
    widgets.HTML("<p style='color:#52757a;font-size:12px'>"
                 "Salida: Mi unidad / MEIKA_Vox / Proyectos / nombre_del_proyecto / "
                 "Transcripciones. Mantén Colab conectado durante la ejecución.</p>"),
    widgets.HTML("<h4>Detalles de ejecución</h4>"),
    log,
]))
